# Запуск пайплайна в Google Colab

**Каждый раз, когда открываете ноутбук или Colab перезапустился**, выполните ячейки 1–3 по порядку: подключение Диска, пути и свежая версия кода. Потом запускайте нужный шаг.

Colab удаляет свою среду, если ноутбук простаивает больше часа-полутора, а также не позже чем через 12 часов. Старые выводы ячеек при этом остаются на экране, поэтому кажется, что всё на месте. Если шаг пишет «Colab перезапустился», просто начните с ячейки 1.

Данные и результаты сохраняются на Google Диск и при перезапуске не теряются. Код каждый раз скачивается заново с GitHub — так всегда используется последняя версия.

Открывайте ноутбук по ссылке из репозитория, а не из сохранённой копии: в копии останутся старые ячейки.

## 1. Подключить Google Диск

Появится окно с запросом доступа — выберите аккаунт и разрешите. Дождитесь строки `Mounted at /content/drive`.

Если появится ошибка `Mountpoint must not already contain files`: меню «Среда выполнения» → «Отключиться и удалить среду выполнения», затем снова с ячейки 1. Ничего не удаляйте вручную: внутри `/content/drive` лежат файлы вашего Диска.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Пути к данным и результатам

In [ ]:
import os

REPO_URL = 'https://github.com/vardov337/crypto-news-forecasting.git'
DATA_DIR = '/content/drive/MyDrive/cryptonews-data'
RESULTS_DIR = '/content/drive/MyDrive/cryptonews-results'

assert os.path.ismount('/content/drive'), 'Диск не подключён — выполните ячейку 1'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ['CRYPTONEWS_DATA_DIR'] = DATA_DIR
os.environ['CRYPTONEWS_RESULTS_DIR'] = RESULTS_DIR
print('Данные:', DATA_DIR)
print('Результаты:', RESULTS_DIR)

## 3. Свежая версия кода

В конце должна напечататься версия с датой. Около минуты.

In [ ]:
%cd /content
!rm -rf /content/repo
!git clone -q $REPO_URL /content/repo
%cd /content/repo
!pip install -q -r requirements.txt
!pip install -q -e .
print('Версия:', open('VERSION').read().strip())

---
## Шаги пайплайна

Запускайте только тот шаг, о котором договорились. Повторный запуск любого шага безопасен: скачанное берётся с Диска, а не качается заново.

### Шаг 0. Проверка окружения
Несколько секунд. Фиксирует версии библиотек.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/00_check_env.py

### Шаг 1. Цены Binance
Первый раз 5–15 минут, повторно — 1–2 минуты (архивы уже на Диске).

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/01_download_prices.py

### Шаг 2. Новости ForkLog
Первый раз около часа. Повторный запуск продолжает с места остановки и уже собранное не скачивает — около минуты.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/02_collect_forklog.py

### Шаг 3. Новости CryptoVision
Несколько минут: скачивает набор с Mendeley и 15-минутные свечи для определения часового пояса.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/03_prepare_news_en.py

### Шаг 4. Очистка новостей ForkLog
Несколько секунд, сеть не нужна.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
!cd /content/repo && python scripts/04_prepare_news_ru.py

---
## Посмотреть результаты

Список таблиц для статьи, которые уже готовы.

In [ ]:
import os
assert os.path.isdir('/content/repo/scripts') and os.environ.get('CRYPTONEWS_DATA_DIR'), \
    'Colab перезапустился: выполните ячейки 1–3, потом этот шаг'
import glob
for path in sorted(glob.glob(f'{RESULTS_DIR}/**/*.csv', recursive=True)):
    print(path.replace(RESULTS_DIR + '/', ''))